# Deep Batch Active Learning by Diverse, Uncertain Gradient Lower Bounds

Ash et al. (ICLR 2020).

**What this notebook gives you**
- A small, runnable BADGE package walkthrough.
- Direct demonstrations of gradient embeddings and k-MEANS++ seeding.
- One single-method active-learning loop with a test-accuracy curve.

**Two ways to use this**
- Run the cells as-is to inspect one smoke-sized BADGE experiment.
- Import the package and replace the loader or classifier for your own pool.

**What this notebook does NOT do**
- It is not a reproduction of the paper's benchmark or its empirical claims.
- It runs 5 acquisition rounds instead of the paper's 349-round SVHN protocol.
- It uses a 12,000-example pool instead of the paper's full training pool.
- It uses one seed instead of the paper's five independent repetitions.
- It uses the bundled flattened MLP instead of the paper's benchmark-specific CNN/ResNet/VGG architectures.
- It uses the package's SGD implementation; the paper describes the Adam variant of SGD.
- Its inferred learning rate, epoch cap, and MLP hidden width are system choices, not paper-stated values.
- It does not implement baselines, comparisons, or multi-method benchmark tables.

## Contents

- [0. Install dependencies (first run only)](#sec-0-install-dependencies-first-run-only)
- [1. Setup](#sec-1-setup)
- [2. Parameters](#sec-2-parameters)
  - [Optional: scale up toward the paper protocol](#sec-optional-scale-up-toward-the-paper-protocol)
- [3. The setup pieces](#sec-3-the-setup-pieces)
  - [3.1 Data](#sec-31-data)
  - [3.2 Model](#sec-32-model)
  - [3.3 Training](#sec-33-training)
  - [3.4 Bootstrap labeled set](#sec-34-bootstrap-labeled-set)
- [4. The BADGE method ⭐](#sec-4-the-badge-method)
  - [4.1 Intuition](#sec-41-intuition)
  - [4.2 Hallucinated gradient embeddings ⭐](#sec-42-hallucinated-gradient-embeddings)
  - [4.3 Diverse batch seeding](#sec-43-diverse-batch-seeding)
  - [4.4 Putting it together](#sec-44-putting-it-together)
- [5. Running active learning end-to-end](#sec-5-running-active-learning-end-to-end)
  - [5.1 The acquisition loop](#sec-51-the-acquisition-loop)
  - [5.2 Learning curve](#sec-52-learning-curve)
- [6. Use your own data](#sec-6-use-your-own-data)

<a id="sec-0-install-dependencies-first-run-only"></a>

## 0. Install dependencies (first run only)

Run this cell once in a new environment. `%pip` installs into the active notebook kernel.

In [1]:
%pip install -r requirements.txt

…[2022 chars stripped for review]…
dy satisfied: python-dateutil>=2.7 in /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages (from matplotlib>=3.7->-r requirements.txt (line 3)) (2.9.0.post0)


…[4664 chars stripped for review]…
ipywidgets->jupyter>=1.0->-r requirements.txt (line 4)) (3.0.16)


…[7012 chars stripped for review]…
 satisfied: jupyterlab-pygments in /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages (from nbconvert->jupyter>=1.0->-r requirements.txt (line 4)) (0.3.0)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


<a id="sec-1-setup"></a>

## 1. Setup

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

from method import (
    compute_gradient_embeddings,
    kmeans_plus_plus_seeding,
    select_batch,
    build_model,
    train_from_scratch,
    load_data,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

<a id="sec-2-parameters"></a>

## 2. Parameters

The rendered table separates values stated by the paper from smoke-scale system defaults and inferred runtime choices. The paper's batch size and initial labeled count are retained; the loop and pool are intentionally smaller.

| Parameter | Variable from paper | Value from paper | Paper value | System value | Where in paper | Used? | Notes |
|---|:-:|:-:|---|---|---|:-:|---|
| `batch_size` | ✅ | ✅ | 100 | — | Experimental setup | ✅ | Paper uses batch_size=100 per data_setup. |
| `num_rounds` | ✅ | ✅ | 349 | 5 | — | ✅ | Paper runs 349 rounds (~34,900 labels at batch_size=100). At smoke scale we... |
| `initial_labeled` | ✅ | ✅ | 100 | — | Experimental setup | ✅ | Paper bootstraps with 100 uniformly-random labeled examples. |
| `pool_size` | ✅ | ✅ | 'full training set' | 12000 | — | ✅ | Paper uses the full training set (typically 60k–73k samples) as the... |
| `learning_rate` | ❌ | ❌ | — | 0.001 | — | ✅ | Paper does not explicitly specify the learning rate... |
| `max_epochs` | ❌ | ❌ | — | 8 | — | ✅ | Paper trains to an explicit training-accuracy threshold with no stated epoch... |
| `train_until_accuracy` | ✅ | ✅ | 0.99 | — | Section 4 EXPERIMENTS | ✅ | Paper-stated training threshold: 'models using cross-entropy loss and the... |
| `hidden_dim` | ❌ | ❌ | — | 256 | — | ✅ | Paper states no data-type-keyed MLP width. We use hidden_dim=256 as the... |

In [3]:
params = {
    "batch_size": {
        "value": 100,
        "source": 'paper',
        "paper_section": 'Experimental setup',
        "note": 'Paper uses batch_size=100 per data_setup.',
    },
    "num_rounds": {
        "value": 5,
        "source": 'system_default',
        "paper_value": 349,
        "reasoning": (
            "Paper runs 349 rounds (~34,900 labels at batch_size=100). At smoke "
            "scale we run 5 rounds at runtime batch_size=100 (~500 labels) — enough "
            "to see a learning curve emerge while keeping the acquisition loop "
            "(which re-trains and re-scores the pool every round) cheap to run on a "
            "laptop CPU at smoke scale."
        ),
        "unused_by_method": True,
    },
    "initial_labeled": {
        "value": 100,
        "source": 'paper',
        "paper_section": 'Experimental setup',
        "note": 'Paper bootstraps with 100 uniformly-random labeled examples.',
        "unused_by_method": True,
    },
    "pool_size": {
        "value": 12000,
        "source": 'system_default',
        "paper_value": 'full training set',
        "reasoning": (
            "Paper uses the full training set (typically 60k–73k samples) as the "
            "unlabeled pool. We subsample to 12,000 to keep each round's "
            "acquisition-scoring pass over the whole pool cheap (the pool is "
            "re-scored every round, so this is the dominant cost of the acquisition "
            "loop). Raised further to 12,000 to satisfy the taxonomy "
            "smoke_economics max_budget_to_pool_ratio floor (0.05); a smaller pool "
            "would over-consume the unlabeled set and weaken diversity-sensitive "
            "acquisition checks. Note: this is a smoke-scale default, not a runtime "
            "estimate — the smoke gate is what actually verifies the notebook fits "
            "its per-cell time budget. Pool ratio (total_budget / pool_size) ≈ "
            "0.05. This is at or below the taxonomy smoke_economics "
            "`max_budget_to_pool_ratio` of 0.05 — diversity baselines are still "
            "meaningfully tested at this scale."
        ),
    },
    "learning_rate": {
        "value": 0.001,
        "source": 'system_inferred',
        "reasoning": (
            "Paper does not explicitly specify the learning rate "
            "(spec.training.learning_rate: 'The paper uses benchmark-specific "
            "training settings.'). Selected 0.001 as the Adam-default convention."
        ),
    },
    "max_epochs": {
        "value": 8,
        "source": 'system_inferred',
        "reasoning": (
            "Paper trains to an explicit training-accuracy threshold with no stated "
            "epoch cap. At smoke scale the labeled sets are tiny (≤ ~500 examples) "
            "and `train_until_accuracy` usually stops well before the cap; 8 is a "
            "low safety bound that keeps per-round training fast at smoke scale. "
            "Increase it for full-scale training."
        ),
    },
    "train_until_accuracy": {
        "value": 0.99,
        "source": 'paper',
        "paper_section": 'Section 4 EXPERIMENTS',
        "note": (
            "Paper-stated training threshold: 'models using cross-entropy loss and "
            "the Adam variant of SGD until training accuracy exceeds 99%'."
        ),
    },
    "hidden_dim": {
        "value": 256,
        "source": 'system_inferred',
        "reasoning": (
            "Paper states no data-type-keyed MLP width. We use hidden_dim=256 as "
            "the taxonomy-typical AL convention."
        ),
    },
}


def unpack(p: dict) -> dict:
    """Strip provenance and return a flat name -> value dict."""
    return {k: v["value"] for k, v in p.items() if v.get("used_in_notebook", True)}


cfg = unpack(params)

<a id="sec-optional-scale-up-toward-the-paper-protocol"></a>

### Optional: scale up toward the paper protocol

The following is intentionally commented out. It shows the main runtime changes needed for a larger run; paper-faithful results also require the paper's benchmark data, architecture, optimizer protocol, and repeated seeds.

In [4]:
# cfg.update({
#     "num_rounds": 349,
#     "pool_size": 60000,  # example full-scale image-pool order of magnitude
#     "learning_rate": 0.001,  # paper's image-data value; verify for your benchmark
#     "max_epochs": 50,
#     "hidden_dim": 256,
# })

<a id="sec-3-the-setup-pieces"></a>

## 3. The setup pieces

BADGE sits on standard pool-based supervised classification: labeled data, a differentiable classifier, and retraining between queries. These pieces are protocol rather than the paper's contribution; the contribution is the uncertainty-aware final-layer gradient embedding and batch k-MEANS++ selection in §4. The classifier must expose `forward_with_embedding`, returning logits and the real penultimate features used by the final linear layer.

<a id="sec-31-data"></a>

### 3.1 Data

`load_data` returns float32 feature tensors and int64 class-id tensors. The v2 runtime contract declares pool examples as `(N_pool, D)` and test examples as `(N_test, D)`; this loader supplies flattened MNIST features, with `D=784` for the bundled smoke data.

In [5]:
x_pool, y_pool, x_test, y_test = load_data(
    pool_size=cfg["pool_size"], n_test=1000, seed=SEED
)
n_classes = int(y_pool.max().item()) + 1
input_dim = int(x_pool.shape[1])
print("pool:", tuple(x_pool.shape), tuple(y_pool.shape))
print("test:", tuple(x_test.shape), tuple(y_test.shape))
print("input_dim:", input_dim, "classes:", n_classes)

pool: (12000, 784) (12000,)
test: (1000, 784) (1000,)
input_dim: 784 classes: 10


<a id="sec-32-model"></a>

### 3.2 Model

The bundled `GradientEmbeddingClassifier` is a compact MLP with a final linear classifier and an explicit penultimate-feature hook. The paper evaluates several architectures; this flattened MLP is a runnable demo choice. Replacing it is safe only when the replacement preserves the declared `(batch, D) -> (batch, C)` logits path and `forward_with_embedding` contract.

In [6]:
model = build_model(
    input_dim=input_dim,
    n_classes=n_classes,
    hidden_dim=cfg["hidden_dim"],
)
print(model)

GradientEmbeddingClassifier(
  (encoder): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
  )
  (classifier): Linear(in_features=256, out_features=10, bias=True)
)


<a id="sec-33-training"></a>

### 3.3 Training

Algorithm 1 retrains from scratch after each acquisition. The paper describes cross-entropy training until training accuracy exceeds 99% (**Algorithm 1, Section 4**). This package's generated training function uses the supplied SGD protocol and the configured smoke epoch cap; the paper's Adam wording is listed as a departure in the title cell.

<a id="sec-34-bootstrap-labeled-set"></a>

### 3.4 Bootstrap labeled set

BADGE begins with a uniformly random labeled set (**Algorithm 1**). We retain the paper's `initial_labeled` count, train a warmup model from scratch, and reuse that fixed model snapshot only for the §4 helper demonstrations. The end-to-end loop rebuilds a fresh model for every round.

In [7]:
rng = np.random.default_rng(SEED)
bootstrap_labeled_idx = rng.choice(
    len(x_pool), size=cfg["initial_labeled"], replace=False
).astype(np.int64)
bootstrap_model = build_model(
    input_dim=input_dim,
    n_classes=n_classes,
    hidden_dim=cfg["hidden_dim"],
)
bootstrap_model = train_from_scratch(
    bootstrap_model,
    x_pool[bootstrap_labeled_idx],
    y_pool[bootstrap_labeled_idx],
    learning_rate=cfg["learning_rate"],
    max_epochs=cfg["max_epochs"],
    train_until_accuracy=cfg["train_until_accuracy"],
    seed=SEED,
)
bootstrap_model.eval()
with torch.no_grad():
    bootstrap_loss = torch.nn.functional.cross_entropy(
        bootstrap_model(x_pool[bootstrap_labeled_idx]),
        y_pool[bootstrap_labeled_idx],
    ).item()
print(f"bootstrap training loss: {bootstrap_loss:.4f}")

bootstrap training loss: 0.1931


<a id="sec-4-the-badge-method"></a>

## 4. The BADGE method ⭐

BADGE is the paper's contribution (**Algorithm 1, Section 3**). It computes one hallucinated final-layer gradient embedding per unlabeled example, then samples a diverse batch in that gradient space.

<a id="sec-41-intuition"></a>

### 4.1 Intuition

A label is unavailable when the pool is scored, so BADGE uses the model's predicted class as a hallucinated label. The resulting last-layer gradient is small for confident examples and larger when a possible label would induce a larger update (**Section 3, Proposition 1**). Its direction also retains penultimate representation information. k-MEANS++ then favors points far from already selected centers, combining uncertainty and diversity without replacing the gradient embedding with entropy, margins, raw probabilities, or input-space features (**Section 3, Equation 1**).

<a id="sec-42-hallucinated-gradient-embeddings"></a>

### 4.2 Hallucinated gradient embeddings ⭐

`compute_gradient_embeddings` implements the per-example representation from **Eq. 1, Section 3** (`eq-gradient-embedding`, `eq-gradient-block`). For class block `i`, it computes `(p_i - I(yhat=i)) z(x; V)`, where `p` is the softmax output and `z` is the penultimate feature. The code below inspects embedding magnitudes; it does not substitute those magnitudes for the embeddings used by acquisition.

<!-- derived-block: begin {"kind": "implementation_note", "spec": {"element_ids": ["concept-gradient-diversity", "concept-gradient-uncertainty", "concept-hallucinated-label", "concept-quality-diversity-tradeoff", "eq-gradient-block", "eq-gradient-embedding", "eq-softmax-definition", "eq-softmax-model"], "file": "method/method.py", "qualname": "compute_gradient_embeddings"}} -->
**How this is implemented:** [`compute_gradient_embeddings`](method/method.py) (`method/method.py:21`) implements [Diversity in gradient space](METHOD.md#concept-gradient-diversity), [Gradient magnitude as uncertainty](METHOD.md#concept-gradient-uncertainty), [Hallucinated-label uncertainty](METHOD.md#concept-hallucinated-label), [Hyperparameter-free quality and diversity tradeoff](METHOD.md#concept-quality-diversity-tradeoff), [Gradient embedding block formula](METHOD.md#eq-gradient-block), [Hallucinated last-layer gradient embedding](METHOD.md#eq-gradient-embedding), [Softmax activation](METHOD.md#eq-softmax-definition), [Softmax output model](METHOD.md#eq-softmax-model).

<details>
<summary>Source of <code>compute_gradient_embeddings</code> (34 lines)</summary>

**Source: `compute_gradient_embeddings` — method/method.py:21**

```python
def compute_gradient_embeddings(model: Any, x_unlabeled: Any) -> np.ndarray:
    """Compute one hallucinated final-layer gradient per pool example.

    **Eq. 1, Section 3**: for probabilities ``p`` and penultimate features
    ``z``, the class block is ``(p_i - 1[y_hat=i]) * z``.  The result is a
    NumPy array with shape ``(pool_size, class_count * feature_width)``.
    ``model`` must expose ``forward_with_embedding`` returning logits and the
    actual penultimate features immediately before its final linear layer.
    """
    # paper-element: eq-gradient-embedding
    # paper-element: eq-gradient-block
    # paper-element: concept-hallucinated-label
    # essential: Final-layer gradient embedding access
    was_training = bool(model.training)
    model.eval()
    try:
        with torch.no_grad():
            logits, features = model.forward_with_embedding(x_unlabeled)
            # paper-element: eq-softmax-model
            # paper-element: eq-softmax-definition
            probabilities = F.softmax(logits, dim=-1)
            predicted_labels = probabilities.argmax(dim=-1)
            residual = probabilities.clone()
            residual.scatter_(1, predicted_labels.unsqueeze(1),
                              residual.gather(1, predicted_labels.unsqueeze(1)) - 1.0)
            embeddings = (residual.unsqueeze(-1) * features.unsqueeze(1)).reshape(
                features.shape[0], -1
            )
            # paper-element: concept-gradient-uncertainty
            # paper-element: concept-gradient-diversity
            # paper-element: concept-quality-diversity-tradeoff
            return embeddings.detach().cpu().numpy()
    finally:
        model.train(was_training)
```

</details>
<!-- derived-block: end -->

In [8]:
bootstrap_embeddings = compute_gradient_embeddings(
    bootstrap_model, x_pool
)
embedding_norms = np.linalg.norm(bootstrap_embeddings, axis=1)
print("embedding matrix:", bootstrap_embeddings.shape)
print("mean embedding norm:", float(embedding_norms.mean()))
plt.figure(figsize=(6, 3))
plt.hist(embedding_norms, bins=30)
plt.xlabel("final-layer gradient-embedding norm")
plt.ylabel("pool examples")
plt.title("BADGE gradient-space magnitudes")
plt.show()

embedding matrix: (12000, 2560)
mean embedding norm: 4.813790798187256


<a id="sec-43-diverse-batch-seeding"></a>

### 4.3 Diverse batch seeding

`kmeans_plus_plus_seeding` chooses the first center uniformly and subsequent centers with probability proportional to squared distance from the nearest selected center (**Appendix A, Algorithm 2**, `alg-kmeans-plus-plus`). This is the diversity step applied to the actual gradient embeddings.

<!-- derived-block: begin {"kind": "implementation_note", "spec": {"element_ids": ["alg-kmeans-plus-plus"], "file": "method/method.py", "qualname": "kmeans_plus_plus_seeding"}} -->
**How this is implemented:** [`kmeans_plus_plus_seeding`](method/method.py) (`method/method.py:57`) implements [k-MEANS++ seeding sampler](METHOD.md#alg-kmeans-plus-plus).

<details>
<summary>Source of <code>kmeans_plus_plus_seeding</code> (42 lines)</summary>

**Source: `kmeans_plus_plus_seeding` — method/method.py:57**

```python
def kmeans_plus_plus_seeding(
    embeddings: np.ndarray, batch_size: int, *, seed: int
) -> list[int]:
    """Select unique embedding centers with k-MEANS++ (Appendix A, Algorithm 2).

    The first center is uniform; each later center is sampled proportional to
    squared distance from its nearest selected center.  Distances are updated
    incrementally, avoiding an ``(N, batch_size, D)`` broadcast.
    """
    # paper-element: alg-kmeans-plus-plus
    points = np.asarray(embeddings, dtype=np.float64)
    if points.ndim != 2:
        raise ValueError(f"embeddings must be a 2-D array, got shape {points.shape}")
    n_points = points.shape[0]
    if n_points == 0:
        raise ValueError("cannot select from an empty unlabeled pool")
    if batch_size < 1:
        raise ValueError(f"batch_size must be positive, got {batch_size}")
    # paper-fidelity: smoke-sized pools may be smaller than the requested paper batch.
    count = min(int(batch_size), n_points)
    rng = np.random.default_rng(seed)
    selected = np.empty(count, dtype=np.int64)
    selected[0] = int(rng.integers(n_points))
    nearest_sq = np.sum((points - points[selected[0]]) ** 2, axis=1)
    chosen = {int(selected[0])}
    for position in range(1, count):
        total = float(nearest_sq.sum())
        if not np.isfinite(total) or total <= 0.0:
            remaining = np.array(
                [index for index in range(n_points) if index not in chosen],
                dtype=np.int64,
            )
            selected[position] = int(rng.choice(remaining))
        else:
            probabilities = nearest_sq / total
            selected[position] = int(rng.choice(n_points, p=probabilities))
            while int(selected[position]) in chosen:
                selected[position] = int(rng.choice(n_points, p=probabilities))
        chosen.add(int(selected[position]))
        new_sq = np.sum((points - points[selected[position]]) ** 2, axis=1)
        nearest_sq = np.minimum(nearest_sq, new_sq)
    return selected.tolist()
```

</details>
<!-- derived-block: end -->

In [9]:
demo_positions = kmeans_plus_plus_seeding(
    bootstrap_embeddings, cfg["batch_size"], seed=SEED
)
demo_norms = embedding_norms[np.asarray(demo_positions, dtype=np.int64)]
print("selected unique positions:", len(set(demo_positions)))
print("selected batch size:", len(demo_positions))
print("selected mean embedding norm:", float(demo_norms.mean()))

selected unique positions: 100
selected batch size: 100
selected mean embedding norm: 8.206028938293457


<a id="sec-44-putting-it-together"></a>

### 4.4 Putting it together

The public `select_batch` composition first calls the gradient-embedding helper and then calls k-MEANS++ with the requested batch size (**Algorithm 1, Section 3**). The end-to-end cell below keeps the original unlabeled tensor and maps returned positions back to global pool indices.

<a id="sec-5-running-active-learning-end-to-end"></a>

## 5. Running active learning end-to-end

This is one BADGE run on one dataset and one seed. It follows the paper's retrain-from-scratch round structure, but uses the explicit smoke-scale defaults shown in the rendered provenance table. No baseline or comparison method is run.

<a id="sec-51-the-acquisition-loop"></a>

### 5.1 The acquisition loop

<!-- derived-block: begin {"kind": "implementation_note", "spec": {"element_ids": ["alg-badge"], "file": "method/method.py", "qualname": "select_batch"}} -->
**How this is implemented:** [`select_batch`](method/method.py) (`method/method.py:101`) implements [BADGE batch active learning](METHOD.md#alg-badge).

<details>
<summary>Source of <code>select_batch</code> (10 lines)</summary>

**Source: `select_batch` — method/method.py:101**

```python
def select_batch(model: Any, x_unlabeled: Any, batch_size: int, seed: int) -> list[int]:
    """Select a BADGE batch by composing gradient embeddings and k-MEANS++.

    **Algorithm 1, Section 3**: each unlabeled point is represented by its
    hallucinated-label last-layer gradient embedding, then diverse centers are
    selected with k-MEANS++ seeding.
    """
    # paper-element: alg-badge
    embeddings = compute_gradient_embeddings(model, x_unlabeled)
    return kmeans_plus_plus_seeding(embeddings, batch_size, seed=seed)
```

</details>
<!-- derived-block: end -->

In [10]:
def evaluate_accuracy(eval_model, x, y):
    eval_model.eval()
    with torch.no_grad():
        predictions = eval_model(x).argmax(dim=1)
    return float((predictions == y).float().mean().item())


def train_and_evaluate(current_indices, round_seed):
    round_model = build_model(
        input_dim=input_dim,
        n_classes=n_classes,
        hidden_dim=cfg["hidden_dim"],
    )
    round_model = train_from_scratch(
        round_model,
        x_pool[current_indices],
        y_pool[current_indices],
        learning_rate=cfg["learning_rate"],
        max_epochs=cfg["max_epochs"],
        train_until_accuracy=cfg["train_until_accuracy"],
        seed=round_seed,
    )
    round_model.eval()
    with torch.no_grad():
        loss_value = torch.nn.functional.cross_entropy(
            round_model(x_pool[current_indices]), y_pool[current_indices]
        ).item()
    accuracy = evaluate_accuracy(round_model, x_test, y_test)
    print(
        f"round seed={round_seed}: labels={len(current_indices)}, "
        f"training_loss={loss_value:.4f}, test_accuracy={accuracy:.4f}"
    )
    return round_model, accuracy


# Rebuild all loop state from the stable bootstrap output on every execution.
labeled_idx = np.asarray(bootstrap_labeled_idx, dtype=np.int64).copy()
unlabeled_idx = np.setdiff1d(
    np.arange(len(x_pool), dtype=np.int64), labeled_idx
)
learning_curve = []
label_counts = []

model, accuracy = train_and_evaluate(labeled_idx, SEED)
label_counts.append(len(labeled_idx))
learning_curve.append(accuracy)

for round_index in range(cfg["num_rounds"]):
    model.eval()
    with torch.no_grad():
        selected_positions = select_batch(
            model,
            x_pool[unlabeled_idx],
            cfg["batch_size"],
            SEED + round_index,
        )
    selected_positions = np.asarray(selected_positions, dtype=np.int64)
    chosen_global = unlabeled_idx[selected_positions]
    labeled_idx = np.union1d(labeled_idx, chosen_global)
    unlabeled_idx = np.setdiff1d(unlabeled_idx, chosen_global)
    model, accuracy = train_and_evaluate(
        labeled_idx, SEED + round_index + 1
    )
    label_counts.append(len(labeled_idx))
    learning_curve.append(accuracy)

print("label counts:", label_counts)
print("evaluated points:", len(learning_curve))

round seed=42: labels=100, training_loss=0.1931, test_accuracy=0.7410


round seed=43: labels=200, training_loss=0.2664, test_accuracy=0.7720


round seed=44: labels=300, training_loss=0.3826, test_accuracy=0.8350


round seed=45: labels=400, training_loss=0.4515, test_accuracy=0.8510


round seed=46: labels=500, training_loss=0.5252, test_accuracy=0.8560


round seed=47: labels=600, training_loss=0.5230, test_accuracy=0.8640
label counts: [100, 200, 300, 400, 500, 600]
evaluated points: 6


<a id="sec-52-learning-curve"></a>

### 5.2 Learning curve

Each point is evaluated after training on the labeled set shown on the x-axis: the initial bootstrap point, followed by one post-acquisition point for each configured round. This is a single BADGE curve, not a comparison overlay.

In [11]:
plt.figure(figsize=(7, 4))
plt.plot(label_counts, learning_curve, marker="o", label="BADGE")
plt.xlabel("number of labeled examples")
plt.ylabel("test accuracy")
plt.title("BADGE active learning on the smoke dataset")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

<a id="sec-6-use-your-own-data"></a>

## 6. Use your own data

`load_data` accepts a `.pt` file or directory containing `{"x_pool", "y_pool", "x_test", "y_test"}`, a `.json` file with the same keys as nested lists, or a directory containing `pool.csv` (or `train.csv`) and `test.csv` with the integer label in the last column. Pass the path explicitly with `load_data(path="...")`, or place a file in `method/example_data/` and call `load_data()`.

The bundled classifier expects flattened 2-D feature tensors. For image-shaped inputs, swap in a differentiable architecture whose logits and `forward_with_embedding` penultimate features satisfy the model contract. Keep BADGE's final-layer gradient embeddings and k-MEANS++ selection unchanged.